# 第4回　離散確率分布：二項分布・ポアソン分布
## ―― 「独立な試行」を数式にする

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

第2回で定義した **独立**。今日はその「独立な試行」を $n$ 回くりかえすと結果がどんな分布になるかを学ぶ。それが **二項分布** と **ポアソン分布** だ。▶ を上から押そう。

最後に「もし試行が独立でなかったら分布はどう壊れるか」を見て、今期の背骨につなげる。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats
print("準備OK。次のセルへ。")

---
## 1. 直感クイズ ―― 10回投げて7回表。イカサマ？

公正なコイン（表の確率 0.5）を10回投げたら、表が **7回** 出た。

**問い：これは「イカサマだ！」と言えるほど珍しいことか？**

7回は5回より多いけれど、偶然でもよくある範囲？　予想を決めてから ▶。

In [ ]:
# 公正なコイン（p=0.5）を10回投げたときの『表の回数』の確率分布＝二項分布
n, p = 10, 0.5
分布 = stats.binom(n, p)

# 7回以上表が出る確率
p_7以上 = 分布.sf(6)   # sf(6)=P(X>6)=P(X>=7)
print(f"ちょうど7回表が出る確率 ： {分布.pmf(7):.3f}")
print(f"7回以上 表が出る確率　 ： {p_7以上:.3f}")
print(f"→ 約 {p_7以上:.0%}。20回に3〜4回は起きる『よくある偶然』。イカサマと断じるには弱い。")

**7回以上は約17%。** 5〜6回に1回は起きる、ありふれた偶然だ。これだけでイカサマとは言えない。

（ここで「珍しさ」を確率で測る発想は、第8〜9回の **仮説検定・p値** につながる。「珍しい＝偶然では説明しにくい」をどこで線引きするか、が後の大問題になる。）

---
## 2. 二項分布 ―― 独立な試行を n 回

**確率変数**：偶然で値が決まる量（ここでは「10回中の表の回数」）。

成功確率 $p$ の試行を **独立に** $n$ 回くりかえし、成功回数を数えると **二項分布** $\mathrm{Binom}(n,p)$ になる。

$$P(X=k) = \binom{n}{k}\,p^{k}(1-p)^{n-k}, \qquad E[X]=np,\quad V[X]=np(1-p)$$

> ⚠️ **大前提は『独立』**
> 
> 二項分布は **各試行が独立** であることを前提にしている。1回目の結果が2回目に影響しないからこそ、確率を単純にかけ算できる。この前提が崩れると、二項分布は使えない（→このノートの最後で確認）。


実際に1万回シミュレーションして、理論の式と一致するか見よう。


In [ ]:
rng = np.random.default_rng(4)
n, p, 試行 = 10, 0.5, 10000
# 10枚のコインを独立に投げる、を1万セット
表の回数 = (rng.random((試行, n)) < p).sum(axis=1)

plt.figure(figsize=(7, 4))
plt.hist(表の回数, bins=np.arange(-0.5, n + 1.5), density=True,
         color="#bcd", edgecolor="white", label="シミュレーション")
ks = np.arange(0, n + 1)
plt.plot(ks, stats.binom(n, p).pmf(ks), "o-", color="#e8503a", label="理論：二項分布")
plt.xlabel("10回中の表の回数")
plt.ylabel("確率")
plt.title(f"二項分布 Binom(n=10, p=0.5)　平均={n*p:.0f}, 分散={n*p*(1-p):.2f}")
plt.legend()
plt.show()
print(f"シミュレーションの平均 {表の回数.mean():.2f}（理論 {n*p:.0f}） / 分散 {表の回数.var():.2f}（理論 {n*p*(1-p):.2f}）")

---
## 3. ポアソン分布 ―― 稀な事象の件数

「1日にかかってくる迷惑電話の件数」「1ページあたりの誤植の数」「1時間に窓口へ来る客数」――
**たくさんの機会があり、それぞれで起こる確率は小さい** 事象の件数は、**ポアソン分布** $\mathrm{Poisson}(\lambda)$ になる。

$$P(X=k)=\frac{\lambda^{k}e^{-\lambda}}{k!},\qquad E[X]=V[X]=\lambda$$

$\lambda$ は「平均して何件起きるか」。二項分布で $n$ がとても大きく $p$ がとても小さいとき（$\lambda=np$ を一定に保つ）、二項分布はポアソン分布に近づく。

In [ ]:
# 二項分布(n大, p小) が ポアソン分布(λ=np) に近づくのを確認
λ = 3
ks = np.arange(0, 13)
plt.figure(figsize=(7, 4))
for n_big in [10, 50, 500]:
    p_small = λ / n_big
    plt.plot(ks, stats.binom(n_big, p_small).pmf(ks), "o--", alpha=0.6,
             label=f"二項 n={n_big}, p={p_small:.3f}")
plt.plot(ks, stats.poisson(λ).pmf(ks), "s-", color="#e8503a", lw=2,
         label=f"ポアソン λ={λ}")
plt.xlabel("件数 k")
plt.ylabel("確率")
plt.title("n を大きく・p を小さくすると、二項分布はポアソン分布に近づく")
plt.legend()
plt.show()

$n$ が大きくなるほど、二項分布（点線）がポアソン分布（赤）にぴったり重なる。**稀な事象を数えるときはポアソンが便利**、と覚えておけばよい。

| | 二項分布 | ポアソン分布 |
|---|---|---|
| 使う場面 | 試行回数 $n$ がはっきり決まっている（10回投げる） | 機会が膨大で件数だけ分かる（1日の件数） |
| パラメータ | $n,\ p$ | $\lambda$（平均件数） |
| 共通の前提 | **各試行・各事象が独立** | **各事象が独立** |

---
## 4. もし試行が「独立でなかったら」？

二項分布もポアソン分布も、**各試行が独立**であることが前提だった。では、独立が崩れるとどうなるか。

例えば10人が順番に答えるクイズで、各人が **8割の確率で前の人の答えを真似る**（空気を読む）としよう。一人ひとりはコインで決めているつもりでも、互いに影響し合っている。このとき「10人中の正解者数」はどんな分布になる？

予想：独立なときと同じ二項分布のまま？

In [ ]:
rng = np.random.default_rng(40)
n, p, 試行 = 10, 0.5, 20000
真似る確率 = 0.8

def 相関した10人():
    答え = np.empty(n, dtype=bool)
    答え[0] = rng.random() < p
    for i in range(1, n):
        if rng.random() < 真似る確率:
            答え[i] = 答え[i - 1]          # 前の人を真似る（独立が壊れる）
        else:
            答え[i] = rng.random() < p     # たまに自分で決める
    return 答え.sum()

独立な表の回数 = (rng.random((試行, n)) < p).sum(axis=1)
相関した表の回数 = np.array([相関した10人() for _ in range(試行)])

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
bins = np.arange(-0.5, n + 1.5)
axes[0].hist(独立な表の回数, bins=bins, density=True, color="#3949ab", edgecolor="white")
axes[0].set_title(f"独立に判断\n平均{独立な表の回数.mean():.1f} / 分散{独立な表の回数.var():.1f}")
axes[1].hist(相関した表の回数, bins=bins, density=True, color="#e8503a", edgecolor="white")
axes[1].set_title(f"8割で前の人を真似る（独立でない）\n平均{相関した表の回数.mean():.1f} / 分散{相関した表の回数.var():.1f}")
for ax in axes:
    ax.set_xlabel("10人中の『表（正解）』の人数")
axes[0].set_ylabel("確率")
plt.tight_layout()
plt.show()

**平均はどちらも約5で同じ。だが分布の形がまるで違う。**

独立なときは5付近に集まる（二項分布）。ところが真似合うと、**0人や10人といった極端な結果がぐっと増え、ばらつき（分散）が大きく膨らむ**。みんなが同じ方向に流れるからだ。

> 💬 **今期の背骨**
> 
> 独立が崩れると、二項分布という前提が成り立たなくなる。「平均は同じ」でも「ばらつきは激増」する。この『真似ると極端な結果が増える』現象こそ、第12回 **情報カスケード**・第13回 **コンドルセの陪審定理** で集団の判断を狂わせる正体だ。今日はその種を見た。


---
## 今日のまとめ

| 分布 | 何の分布か | 式 | 前提 |
|---|---|---|---|
| 二項分布 | 独立な試行 $n$ 回中の成功回数 | $\binom{n}{k}p^k(1-p)^{n-k}$ | 各試行が独立 |
| ポアソン分布 | 稀な事象の件数 | $\dfrac{\lambda^k e^{-\lambda}}{k!}$ | 各事象が独立 |

- 10回中7回表は約17%＝ありふれた偶然。「珍しさ」を確率で測る発想は第8〜9回（検定・p値）へ。
- 二項・ポアソンはどちらも **独立** が大前提。$n$ が大きく $p$ が小さいと二項→ポアソン。
- **独立が崩れる（真似合う）と、平均は同じでもばらつきが激増**し、極端な結果が増える＝集団意思決定の伏線。

> **課題（Moodle）**：二項/ポアソンの計算と使い分け（自動採点）＋「独立試行という前提が崩れると、二項分布の予測は何を間違えるか」の記述。詳しくはMoodleの第4回課題を見ること。

> **次回予告**：第5回「連続確率分布：正規分布の数理」。なぜ世界は正規分布だらけなのか ―― 答えはまた **独立** にある（多数の独立な要因の和）。